In [1]:
import os
import requests
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("FANTASYPROS_API_KEY")

Check API key loaded. DO NOT print it.

In [2]:
assert API_KEY is not None, "FantasyPros API key not found"
print("FantasyPros API key loaded successfully.")

FantasyPros API key loaded successfully.


Assemple API request components for player data

In [19]:
BASE_URL = "https://api.fantasypros.com/public/v2/json"

url = f"{BASE_URL}/nfl/2026/consensus-rankings"

headers = {
    "x-api-key": API_KEY
}

params = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF"
}

# Inspect, but NOT the API Key
print("URL:", url)
print("Parameters:", params)
print("API key configured:", API_KEY is not None)

URL: https://api.fantasypros.com/public/v2/json/nfl/2026/consensus-rankings
Parameters: {'position': 'ALL', 'type': 'ADP', 'scoring': 'HALF'}
API key configured: True


Make a test API call.  

NOTE - ONLY RUN THIS ONCE! Only 50 API calls/day. 

In [20]:
response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

# Inspect the player data response

In [21]:
data = response.json()

print("Status:", response.status_code)
print("Tier:", data.get("tier"))
print("Public API limited:", data.get("public_api_limited"))
print("Count:", data.get("count"))
print("Limit:", data.get("limit"))
print("Players returned:", len(data.get("players", [])))

Status: 200
Tier: premium
Public API limited: True
Count: 340
Limit: None
Players returned: 340


In [22]:
for key, value in data.items():
    if key != "players":
        print(f"{key}: {value}")

sport: NFL
type: ADP Half PPR
ranking_type_name: adp
year: 2026
week: 0
position_id: ALL
scoring: HALF
filters: 236,439,4350
count: 340
total_experts: 3
last_updated: 8/28
last_updated_ts: 1787901615
public_api_limited: True
tier: premium


In [ ]:
# # Full response data
# data

## Save player data locally

In [ ]:
import json
from pathlib import Path

raw_path = Path("../data/raw/fantasypros_adp_2026_half.json")

with open(raw_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved raw response to: {raw_path}")

Now confirm it exists & saved

In [ ]:
print(raw_path.exists())
print(f"{raw_path.stat().st_size / 1024:.1f} KB")

# Inspect Player Data

In [23]:
print(len(data["players"]))
data["players"][0]

340


{'player_id': 22968,
 'player_name': 'Jahmyr Gibbs',
 'sportsdata_id': 'fef9457e-6497-47de-9bf2-cc3b95929375',
 'player_team_id': 'DET',
 'player_position_id': 'RB',
 'player_positions': 'RB',
 'player_short_name': 'J. Gibbs',
 'player_eligibility': 'RB',
 'player_yahoo_positions': 'RB',
 'player_page_url': 'https://www.fantasypros.com/nfl/players/jahmyr-gibbs.php',
 'player_filename': 'jahmyr-gibbs.php',
 'player_yahoo_id': '40059',
 'cbs_player_id': '3162723',
 'player_bye_week': '6',
 'player_owned_avg': 99.5,
 'player_owned_espn': 99.9,
 'player_owned_yahoo': 100,
 'player_ecr_delta': None,
 'rank_ecr': 1,
 'rank_min': '1',
 'rank_max': '1',
 'rank_ave': '1.00',
 'rank_std': '0.00',
 'pos_rank': 'RB1',
 'tier': 1}

# Inspect ADP source metadata

In [ ]:
experts_url = f"{BASE_URL}/nfl/2026/rankings/experts"

experts_params = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF"
}

print("URL:", experts_url)
print("Parameters:", experts_params)

API CALL! - ONLY RUN ONCE!   
This is for platform-specific ADP data. 

In [ ]:
experts_response = requests.get(
    experts_url,
    headers=headers,
    params=experts_params,
    timeout=30
)

print("Status code:", experts_response.status_code)

In [ ]:
experts_data = experts_response.json()

print(type(experts_data))
print(experts_data.keys())

In [ ]:
print("Expert count:", len(experts_data["experts"]))

In [ ]:
experts_data["experts"]

## Retry Player Data with "Experts" 
ADP Metadata did not return "expert" data across platforms. 

In [ ]:
params_with_experts = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF",
    "experts": "show"
}

API CALL BELOW! - Only run this once

In [ ]:
response_with_experts = requests.get(
    url,                    # original consensus-rankings URL
    headers=headers,
    params=params_with_experts,
    timeout=30
)

In [ ]:
data_with_experts = response_with_experts.json()

print("Status:", response_with_experts.status_code)
print("Filters:", data_with_experts.get("filters"))
print("Total experts:", data_with_experts.get("total_experts"))
print("Expert names:", data_with_experts.get("expert_name"))
print("Experts available:", data_with_experts.get("experts_available"))

In [ ]:
print(data_with_experts.keys())

for key in [
    "expert_name",
    "expert_pub",
    "expert_twitter",
    "experts_available"
]:
    print(key, "->", key in data_with_experts, data_with_experts.get(key))

## Explore source IDs by "Expert" (platform)
Expert 4350 == Sleeper.  

3 API Calls below!

In [ ]:
source_ids = [236, 439, 4350]

source_responses = {}

for source_id in source_ids:
    source_params = {
        "position": "ALL",
        "type": "ADP",
        "scoring": "HALF",
        "filters": str(source_id)
    }

    r = requests.get(
        url,
        headers=headers,
        params=source_params,
        timeout=30
    )

    print(source_id, r.status_code)

    source_responses[source_id] = r.json()

Inspect responses

In [ ]:
for source_id, source_data in source_responses.items():
    print(
        source_id,
        "count:", source_data.get("count"),
        "filters:", source_data.get("filters"),
        "total_experts:", source_data.get("total_experts")
    )

## Still not what we want --> try "experts":"available"

In [ ]:
params_available = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF",
    "experts": "available"
}

In [ ]:
response_available = requests.get(
    url,
    headers=headers,
    params=params_available,
    timeout=30
)

In [ ]:
available_data = response_available.json()

print("Status:", response_available.status_code)
print("Total experts:", available_data.get("total_experts"))
print("Filters:", available_data.get("filters"))
print("Experts available type:", type(available_data.get("experts_available")))
print("Experts available:", available_data.get("experts_available"))

# Now try "ranknings" endpoint

In [ ]:
rankings_url = f"{BASE_URL}/nfl/2026/rankings"

rankings_params = {
    "week": 0
}

print("URL:", rankings_url)
print("Parameters:", rankings_params)

API Call!

In [ ]:
rankings_response = requests.get(
    rankings_url,
    headers=headers,
    params=rankings_params,
    timeout=30
)

print("Status:", rankings_response.status_code)

### Inspect rankings response

In [ ]:
rankings_data = rankings_response.json()

print(type(rankings_data))
print(rankings_data.keys())

In [ ]:
print("experts type:", type(rankings_data["experts"]))
print("players type:", type(rankings_data["players"]))
print("ecr_experts type:", type(rankings_data["ecr_experts"]))

In [ ]:
print("experts count:", len(rankings_data["experts"]))
print("players count:", len(rankings_data["players"]))

In [ ]:
rankings_data["experts"].keys()

In [ ]:
rankings_data["players"][0]

In [ ]:
print(rankings_data["ecr_experts"].keys())

In [ ]:
print(type(rankings_data["experts"]["HALF"]))
rankings_data["experts"]["HALF"]

In [ ]:
print(type(rankings_data["ecr_experts"]["HALF"]))
rankings_data["ecr_experts"]["HALF"]

In [ ]:
gibbs = next(
    player for player in rankings_data["players"]
    if player["player_name"] == "Jahmyr Gibbs"
)

gibbs["rank"]

# Explore 2026 Preseason Projections

In [10]:
BASE_URL = "https://api.fantasypros.com/public/v2/json"

projections_url = f"{BASE_URL}/nfl/2026/projections"

headers = {
    "x-api-key": API_KEY
}

projections_params = {
    "week": 0,
    "position": "ALL",
    "scoring": "HALF"
}

print("URL:", projections_url)
print("Parameters:", projections_params)

URL: https://api.fantasypros.com/public/v2/json/nfl/2026/projections
Parameters: {'week': 0, 'position': 'ALL', 'scoring': 'HALF'}


API CALL! - run once

In [11]:
projections_response = requests.get(
    projections_url,
    headers=headers,
    params=projections_params,
    timeout=30
)

print("Status:", projections_response.status_code)

Status: 200


In [12]:
projections_data = projections_response.json()

print("Response type:", type(projections_data))
print("Top-level keys:", projections_data.keys())

Response type: <class 'dict'>
Top-level keys: dict_keys(['season', 'week', 'count', 'positions', 'scoring', 'experts', 'players', 'public_api_limited', 'tier'])


In [13]:
# Love this cell for understanding API response structure
for key, value in projections_data.items():
    if not isinstance(value, (list, dict)):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {type(value).__name__} with {len(value)} items")

season: 2026
week: 0
count: 603
positions: QB,RB,WR,TE,K,DST
scoring: STD
experts: list with 3 items
players: list with 603 items
public_api_limited: True
tier: premium


In [14]:
projections_data["players"][0]

{'fpid': 17298,
 'mflid': 13589,
 'name': 'Josh Allen',
 'position_id': 'QB',
 'team_id': 'BUF',
 'filename': 'josh-allen-qb.php',
 'stats': {'points': 372.47,
  'points_ppr': 372.47,
  'points_half': 372.47,
  'pass_att': 491.88,
  'pass_cmp': 333.36,
  'pass_yds': 3816.77,
  'pass_tds': 27.42,
  'pass_ints': 11.19,
  'pass_yds_300': 0,
  'pass_yds_400': 0,
  'rush_att': 118.13,
  'rush_yds': 585.97,
  'rush_tds': 11.82,
  'rush_yds_100': 0,
  'rush_yds_200': 0,
  'scrimage_yards_100': 0,
  'scrimage_yards_200': 0,
  'fumbles': 4.1,
  'ret_tds': 0,
  '2pt_tds': 0}}

In [ ]:
[ # Confirms that fpid here == player_id in consensus data
    player for player in projections_data["players"]
    if player.get("name") == "Jahmyr Gibbs"
][0]

{'fpid': 22968,
 'mflid': 16162,
 'name': 'Jahmyr Gibbs',
 'position_id': 'RB',
 'team_id': 'DET',
 'filename': 'jahmyr-gibbs.php',
 'stats': {'points': 301.75,
  'points_ppr': 373.01,
  'points_half': 337.38,
  'rush_att': 274.69,
  'rush_yds': 1382.53,
  'rush_tds': 13.82,
  'rush_yds_100': 0,
  'rush_yds_200': 0,
  'scrimage_yards_100': 0,
  'scrimage_yards_200': 0,
  'rec_rec': 71.26,
  'rec_yds': 580.97,
  'rec_tds': 4.13,
  'rec_yds_100': 0,
  'rec_yds_200': 0,
  'fumbles': 1.13,
  'ret_tds': 0,
  '2pt_tds': 0}}

In [25]:
# Check that ids match acrosss the whole dataset

projection_ids_set = {
    player["fpid"]
    for player in projections_data["players"]
}

player_ids_set = {
    player["player_id"]
    for player in data["players"]
}

match_count = len(player_ids_set & projection_ids_set)
total_count = len(player_ids_set)

match_count, total_count, match_count / total_count


(301, 340, 0.8852941176470588)

In [31]:
# Inspect missing players

import pandas as pd

missing_players_df = pd.DataFrame([
    player
    for player in data["players"]
    if player["player_id"] in missing_projection_ids
])

missing_players_df[
    ["player_name", "player_team_id", "player_position_id", "player_id"]
]

,player_name,player_team_id,player_position_id,player_id
0,Tyreek Hill,FA,WR,15802
1,Noah Brown,FA,WR,16443
2,Nick Vogel,FA,K,19880
3,J.J. Taylor,FA,RB,19438
4,Jonathan Adams Jr.,FA,WR,22818
5,Kareem Hunt,FA,RB,16425
6,Harrison Wallace III,ARI,WR,28143
7,Mecole Hardman Jr.,BUF,WR,18587
8,Jerell Adams,FA,TE,15831
9,Josh Johnson,FA,WR,24212


## Sanity check on Claude's projection data --> src code

The above was sent to Claude to productionalize in src code. Check below

In [32]:
from fantasy_football.extract.fantasypros import (
    fetch_projections,
    save_raw_response,
    PROJECTIONS_RAW_FILENAME,
)

In [ ]:
# API CALL! - run once

projections_dict_raw = fetch_projections()

In [ ]:
print("season:", projections_dict_raw.get("season"))
print("week:", projections_dict_raw.get("week"))
print("count:", projections_dict_raw.get("count"))
print("positions:", projections_dict_raw.get("positions"))
print("tier:", projections_dict_raw.get("tier"))

print(
    "Josh Allen projected HALF:",
    projections_dict_raw["players"][0]["stats"]["points_half"]
)

season: 2026
week: 0
count: 603
positions: QB,RB,WR,TE,K,DST
tier: premium
Josh Allen projected HALF: 372.47


In [ ]:
saved_path = save_raw_response(
    projections_dict_raw,
    filename=PROJECTIONS_RAW_FILENAME
)

print(saved_path)

/Users/braddotson/Desktop/Github/Fantasy_Football_Data_Pipeline/data/raw/fantasypros_projections_2026.json


# Explore 2025 Season-Long Player Points

In [2]:
# Self-contain this ssection with imports and function definitions
import os
import requests
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ["FANTASYPROS_API_KEY"]

BASE_URL = "https://api.fantasypros.com/public/v2/json"

player_points_url = f"{BASE_URL}/nfl/2025/player-points"

headers = {
    "x-api-key": api_key
}

player_points_params = {
    "position": "ALL",
    "scoring": "HALF",
    "start": 1,
    "end": 18,
}

print("URL:", player_points_url)
print("Parameters:", player_points_params)

URL: https://api.fantasypros.com/public/v2/json/nfl/2025/player-points
Parameters: {'position': 'ALL', 'scoring': 'HALF', 'start': 1, 'end': 18}


In [3]:
# API CALL! - run once

player_points_response = requests.get(
    player_points_url,
    headers=headers,
    params=player_points_params,
    timeout=30
)

print("Status:", player_points_response.status_code)

Status: 200


In [4]:
player_points_data = player_points_response.json()

print("Response type:", type(player_points_data))
print("Top-level keys:", player_points_data.keys())

Response type: <class 'dict'>
Top-level keys: dict_keys(['season', 'scoring', 'players', 'public_api_limited', 'tier'])


In [5]:
# Love this cell for understanding API response structure
for key, value in player_points_data.items():
    if not isinstance(value, (list, dict)):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {type(value).__name__} with {len(value)} items")

season: 2025
scoring: HALF
players: list with 2166 items
public_api_limited: True
tier: premium


In [6]:
player_points_data["players"][0]

{'player_id': 8000,
 'player_name': 'Arizona Cardinals',
 'position_id': 'DST',
 'team_id': 'ARI',
 'filename': 'arizona-defense.php',
 'games': 17,
 'points': 77,
 'average': 4.5,
 'weeks': {'1': 5,
  '2': 13,
  '3': 6,
  '4': 5,
  '5': 7,
  '6': 2,
  '7': 1,
  '9': 15,
  '10': 6,
  '11': -1,
  '12': 17,
  '13': 3,
  '14': -4,
  '15': -1,
  '16': 7,
  '17': -1,
  '18': -3}}

In [7]:
[
    player
    for player in player_points_data["players"]
    if player.get("player_id") == 22968
][0]

{'player_id': 22968,
 'player_name': 'Jahmyr Gibbs',
 'position_id': 'RB',
 'team_id': 'DET',
 'filename': 'jahmyr-gibbs.php',
 'games': 17,
 'points': 328.40000000000003,
 'average': 19.3,
 'weeks': {'1': 10,
  '2': 17.9,
  '3': 24.4,
  '4': 16.7,
  '5': 15.7,
  '6': 7,
  '7': 35.3,
  '9': 4.3,
  '10': 36.7,
  '11': 17.1,
  '12': 49.9,
  '13': 10.1,
  '14': 33.5,
  '15': 7.8,
  '16': 17.8,
  '17': 5.4,
  '18': 18.8}}

In [8]:
# Load cached data for ID validation. Load locally to avoid API call
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

consensus_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "fantasypros_consensus_adp_2026_half.json"
)

with consensus_path.open("r", encoding="utf-8") as f:
    consensus_data = json.load(f)

In [9]:
# Check player_id coverage against the 2026 draft-board universe

player_points_ids_set = {
    player["player_id"]
    for player in player_points_data["players"]
}

consensus_player_ids_set = {
    player["player_id"]
    for player in consensus_data["players"]
}

match_count = len(
    consensus_player_ids_set & player_points_ids_set
)

total_count = len(consensus_player_ids_set)

match_count, total_count, match_count / total_count

(272, 340, 0.8)

In [13]:
import pandas as pd
# Inspect missing players
missing_2025_points_ids = (
    consensus_player_ids_set - player_points_ids_set
)

missing_2025_players_df = pd.DataFrame([
    player
    for player in consensus_data["players"]
    if player["player_id"] in missing_2025_points_ids
])

missing_2025_players_df[
    [
        "player_name",
        "player_team_id",
        "player_position_id",
        "rank_ecr",
        "pos_rank",
    ]
]

,player_name,player_team_id,player_position_id,rank_ecr,pos_rank
0,Jeremiyah Love,ARI,RB,26,RB13
1,Jadarian Price,SEA,RB,59,RB26
2,Carnell Tate,TEN,WR,74,WR31
3,Jonathon Brooks,CAR,RB,82,RB32
4,Makai Lemon,PHI,WR,101,WR39
...,...,...,...,...,...
63,Equanimeous St. Brown,FA,WR,335,WR111
64,Kearis Jackson,FA,WR,336,WR112
65,Barika Kpeenu,TB,RB,337,RB93
66,N'Keal Harry,FA,WR,338,WR113


In [14]:
missing_2025_players_df[
    ["player_name", "rank_ecr", "pos_rank", "player_team_id"]
].sort_values("rank_ecr").head(30)

,player_name,rank_ecr,pos_rank,player_team_id
0,Jeremiyah Love,26,RB13,ARI
1,Jadarian Price,59,RB26,SEA
2,Carnell Tate,74,WR31,TEN
3,Jonathon Brooks,82,RB32,CAR
4,Makai Lemon,101,WR39,PHI
5,Jordyn Tyson,107,WR42,NO
6,De'Zhaun Stribling,113,WR47,SF
7,KC Concepcion,127,WR49,CLE
8,Mike Washington Jr.,128,RB42,LV
9,Jonah Coleman,146,RB48,DEN


In [15]:
pd.cut(
    missing_2025_players_df["rank_ecr"],
    bins=[0, 50, 100, 150, 200, 250, 300, 350],
).value_counts().sort_index()

rank_ecr
(0, 50]        1
(50, 100]      3
(100, 150]     7
(150, 200]     4
(200, 250]    12
(250, 300]    19
(300, 350]    22
Name: count, dtype: int64

## Sanity check on Claude's 2025 performance data --> src code

The above was sent to Claude to productionalize in src code. Check below

In [16]:
from fantasy_football.extract.fantasypros import (
    fetch_player_points,
    save_raw_response,
    PLAYER_POINTS_RAW_FILENAME,
)

player_points_dict_raw = fetch_player_points()

print("season:", player_points_dict_raw.get("season"))
print("scoring:", player_points_dict_raw.get("scoring"))
print("player records:", len(player_points_dict_raw.get("players", [])))

season: 2025
scoring: HALF
player records: 2166


In [17]:
# Looks good. Save to local
saved_path = save_raw_response(
    player_points_dict_raw,
    filename=PLAYER_POINTS_RAW_FILENAME,
)

print(saved_path)

/Users/braddotson/Desktop/Github/Fantasy_Football_Data_Pipeline/data/raw/fantasypros_player_points_2025_half.json
